Notebook-Setup und zentrale Imports

In [ ]:
import os
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import pickle

# Seed für Reproduzierbarkeit
np.random.seed(42)
tf.random.set_seed(42)

RANDOM_STATE = 42
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Projektpfade relativ zum Notebooks-Ordner
repo_root = pathlib.Path().resolve().parent
data_dir = repo_root / "data"
models_dir = repo_root / "models"
splits_csv = data_dir / "splits"
data_path = data_dir / "processed"

models_dir.mkdir(parents=True, exist_ok=True)
splits_csv.mkdir(parents=True, exist_ok=True)
data_path.mkdir(parents=True, exist_ok=True)

## 1. Daten laden

Bilder laden, auf 224x224 RGB skalieren und Labels sammeln

In [ ]:
# Splits laden
train_split = pd.read_csv(splits_csv / "train.csv")
val_split = pd.read_csv(splits_csv / "val.csv")
test_split = pd.read_csv(splits_csv / "test.csv")

print("Train split shape:", train_split.shape)
print("Val split shape:", val_split.shape)
print("Test split shape:", test_split.shape)
print("\nFirst rows of train split:")
print(train_split.head())

In [ ]:
# tf.data Pipeline für das CNN
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),  
    layers.RandomContrast(0.1), 
    layers.RandomZoom(0.05),
], name="data_augmentation")


def decode_and_resize(path, label):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_png(img_bytes, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label


def augment_and_clip(img, label):
    img = data_augmentation(img, training=True)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label


def make_dataset(df: pd.DataFrame, labels: np.ndarray, training: bool = False) -> tf.data.Dataset:
    if len(df) == 0:
        raise ValueError("Leerer Split erkannt – prüfe train/val/test CSVs und 'path_processed'.")
    paths = df["path_processed"].astype(str).to_numpy()
    labels = labels.astype(np.int64)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()  # reduziert Disk-I/O und stabilisiert Loss-Kurven

    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=RANDOM_STATE, reshuffle_each_iteration=True)
        ds = ds.map(augment_and_clip, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

Label-Encoding auf Basis aller Splits durchführen

In [ ]:
# Label-Encoding und Dataset-Erstellung
le = LabelEncoder()
le.fit(pd.concat([train_split["label"], val_split["label"], test_split["label"]], axis=0))


y_train_enc = le.transform(train_split["label"])
y_val_enc   = le.transform(val_split["label"])
y_test_enc  = le.transform(test_split["label"])


num_classes = len(le.classes_)
print("Klassen:", list(le.classes_))
print("Anzahl Klassen:", num_classes)


train_ds = make_dataset(train_split, y_train_enc, training=True)
val_ds   = make_dataset(val_split,   y_val_enc,   training=False)
test_ds  = make_dataset(test_split,  y_test_enc,  training=False)


# Stichprobe prüfen + Label-Sanity
batch_imgs, batch_labels = next(iter(train_ds))
print("Batch images shape:", batch_imgs.shape)
print("Batch labels shape:", batch_labels.shape)
print("Pixel range min/max:", float(tf.reduce_min(batch_imgs)), float(tf.reduce_max(batch_imgs)))


min_lbl = int(tf.reduce_min(batch_labels))
max_lbl = int(tf.reduce_max(batch_labels))
assert 0 <= min_lbl and max_lbl < num_classes, f"Labels außerhalb des Bereichs [0, {num_classes}): min={min_lbl}, max={max_lbl}"
print(f"Label range OK: [{min_lbl}, {max_lbl}] < {num_classes}")

## 2. Daten normalisieren und in One-Hot Format konvertieren

Bilder skalieren/reshapen und Labels one-hot kodieren

# Sanity-Checks: Labels, Shapes und Klassenanzahl
In diesem Abschnitt prüfen wir, ob die One-Hot-Labels korrekt sind (jede Zeile summiert zu 1), ob die Shapes von Bildern und Labels übereinstimmen und ob die Klassenanzahl konsistent ist.

## 3. Daten explorieren und visualisieren

Übersicht zu Shapes und Klassenverteilung

In [ ]:
# Zusammenfassung der Datensätze
print('Train samples:', len(train_split))
print('Val samples:', len(val_split))
print('Test samples:', len(test_split))


print('Batch example images shape:', batch_imgs.shape)
print('Batch example labels shape:', batch_labels.shape)


# Klassenverteilung anschauen
class_counts = train_split["label"].value_counts()
print("\nKlassenverteilung im Training:")
print(class_counts)

Beispielbilder pro Klasse anzeigen

In [ ]:
# Beispielbilder visualisieren (erste Batch)
fig, ax = plt.subplots(2, 5, figsize=(15, 6))
ax = ax.flatten()


for i in range(min(10, batch_imgs.shape[0])):
    img = batch_imgs[i].numpy()
    ax[i].imshow(img)
    class_name = le.classes_[int(batch_labels[i].numpy())]
    ax[i].set_title(class_name)
    ax[i].axis('off')


plt.tight_layout()
plt.show()

## 4. PlotLosses Callback für Live-Visualisierung

Callback zum Live-Plot von Loss/Val-Loss

In [ ]:
class PlotLosses(keras.callbacks.Callback):
    def on_train_begin(self, logs={}):
        self.i = 0
        self.x = []
        self.losses = []
        self.val_losses = []
        
        self.fig = plt.figure()
        
        self.logs = []

    def on_epoch_end(self, epoch, logs={}):
        
        self.logs.append(logs)
        self.x.append(self.i)
        self.losses.append(logs.get('loss'))
        self.val_losses.append(logs.get('val_loss'))
        self.i += 1
        
        clear_output(wait=True)
        plt.plot(self.x, self.losses, label="loss")
        plt.plot(self.x, self.val_losses, label="val_loss")
        plt.legend()
        plt.show()

plot_losses = PlotLosses()

## 5. Einfaches CNN Modell

Kompaktes CNN mit Adam-Optimizer definieren

In [ ]:
# Modell initialisieren
modelCNN_plankton = Sequential()


# Modell aufsetzen
modelCNN_plankton.add(layers.Conv2D(32,                           # Anzahl der Filter
                                   kernel_size=(3, 3),            # Dimensionen des Fensters
                                   strides=(1, 1),                # Größe der Schritte
                                   padding='same',                # Padding: same=dim(in)=dim(out), valid=no Padding
                                   activation='relu',             # Aktivierungsfunktion
                                   input_shape=(IMG_SIZE, IMG_SIZE, 3)))
modelCNN_plankton.add(layers.BatchNormalization())


modelCNN_plankton.add(layers.MaxPooling2D(pool_size=(2, 2),
                                         strides=(2, 2)))         # MaxPooling Fenstergröße 2x2


modelCNN_plankton.add(layers.Conv2D(64, kernel_size=(3, 3),
                                   strides=(1, 1),
                                   padding='same',
                                   activation='relu'))
modelCNN_plankton.add(layers.BatchNormalization())


modelCNN_plankton.add(layers.MaxPooling2D(pool_size=(2, 2),
                                         strides=(2, 2)))


modelCNN_plankton.add(layers.Flatten())                            # Feature-Vektor erzeugen
modelCNN_plankton.add(layers.Dense(128, activation='relu'))
modelCNN_plankton.add(layers.Dense(num_classes, activation='softmax'))  # Output Layer


# Optimizer (reduzierte LR + Gradient Clipping)
myAdam = keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999, clipnorm=1.0)


# Modell kompilieren
modelCNN_plankton.compile(optimizer=myAdam,
                         loss='sparse_categorical_crossentropy',
                         metrics=['accuracy'])


print("Model Summary:")
modelCNN_plankton.summary()

In [ ]:
# Initialer Loss-Check (~log(K)) vor dem Training
expected_init = float(np.log(num_classes))
init_loss_plankton, init_acc_plankton = modelCNN_plankton.evaluate(train_ds, verbose=0)
print(f"Base CNN initial loss: {init_loss_plankton:.4f} (expected ~{expected_init:.4f}), acc: {init_acc_plankton:.4f}")
print("Output units:", getattr(modelCNN_plankton.layers[-1], 'units', None), "Loss:", modelCNN_plankton.loss)
assert modelCNN_plankton.output_shape[-1] == num_classes

Training des Basis-CNN mit Adam (kurz für schnellen Test)

In [ ]:
# Modell trainieren
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-6, verbose=1)


history_cnn = modelCNN_plankton.fit(train_ds,
                                   validation_data=val_ds,
                                   epochs=10,  # kürzer zum schnellen Testen
                                   verbose=1,
                                   callbacks=[plot_losses, early_stopping, reduce_lr])

Evaluation des Basis-CNN auf dem Test-Set

In [ ]:
# Test Evaluation
test_loss, test_accuracy = modelCNN_plankton.evaluate(test_ds)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## 6. CNN mit Dropout-Regularisierung

Dropout-CNN mit SGD (LR-Schedule) definieren

In [ ]:
# Modell mit Dropout initialisieren
modelCNN_dropout = Sequential()


modelCNN_dropout.add(layers.Conv2D(32, kernel_size=(3, 3),
                                  strides=(1, 1),
                                  padding='same',
                                  activation='relu',
                                  input_shape=(IMG_SIZE, IMG_SIZE, 3)))
modelCNN_dropout.add(layers.BatchNormalization())


modelCNN_dropout.add(layers.Dropout(0.1))  # 10% Dropout


modelCNN_dropout.add(layers.MaxPooling2D(pool_size=(2, 2),
                                        strides=(2, 2)))


modelCNN_dropout.add(layers.Conv2D(64, kernel_size=(3, 3),
                                  strides=(1, 1),
                                  padding='same',
                                  activation='relu'))
modelCNN_dropout.add(layers.BatchNormalization())


modelCNN_dropout.add(layers.Dropout(0.1))  # 10% Dropout


modelCNN_dropout.add(layers.MaxPooling2D(pool_size=(2, 2),
                                        strides=(2, 2)))


modelCNN_dropout.add(layers.Flatten())
modelCNN_dropout.add(layers.Dense(128, activation='relu'))
modelCNN_dropout.add(layers.Dropout(0.25))  # 25% Dropout vor Output
modelCNN_dropout.add(layers.Dense(num_classes, activation='softmax'))


# Optimizer mit Learning Rate Schedule (konservativer + Gradient Clipping)
from tensorflow.keras.optimizers.schedules import ExponentialDecay


lr_schedule = ExponentialDecay(
    initial_learning_rate=0.0002,  # vorher 0.0005
    decay_steps=100000,
    decay_rate=0.96,
    staircase=False
)


# Verwende keras.optimizers.SGD mit clipnorm zur Stabilisierung
sgd = keras.optimizers.SGD(learning_rate=lr_schedule, momentum=0.9, nesterov=True, clipnorm=1.0)


# Modell kompilieren
modelCNN_dropout.compile(optimizer=sgd,
                        loss='sparse_categorical_crossentropy',
                        metrics=['accuracy'])


print("Model with Dropout Summary:")
modelCNN_dropout.summary()

In [ ]:
# Initialer Loss-Check (~log(K)) vor dem Training
expected_init = float(np.log(num_classes))
init_loss_dropout, init_acc_dropout = modelCNN_dropout.evaluate(train_ds, verbose=0)
print(f"Dropout CNN initial loss: {init_loss_dropout:.4f} (expected ~{expected_init:.4f}), acc: {init_acc_dropout:.4f}")
print("Output units:", getattr(modelCNN_dropout.layers[-1], 'units', None), "Loss:", modelCNN_dropout.loss)
assert modelCNN_dropout.output_shape[-1] == num_classes

Training des Dropout-CNN mit SGD und LR-Schedule

In [ ]:
# Modell trainieren
history_dropout = modelCNN_dropout.fit(train_ds,
                                      validation_data=val_ds,
                                      epochs=10,  # kürzer zum schnellen Testen
                                      verbose=1,
                                      callbacks=[plot_losses])

Evaluation des Dropout-CNN auf dem Test-Set

In [ ]:
# Test Evaluation
test_loss, test_accuracy = modelCNN_dropout.evaluate(test_ds)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## 7. Model Evaluation und Vergleich

Classification Report und Confusion Matrix erzeugen

In [ ]:
# Predictions und Classification Report
y_pred_cnn = modelCNN_dropout.predict(test_ds)
y_pred_classes = np.argmax(y_pred_cnn, axis=1)
y_true_classes = np.concatenate([y.numpy() for _, y in test_ds])


print("Classification Report - CNN with Dropout:")
print(classification_report(y_true_classes, y_pred_classes, target_names=le.classes_))


# Confusion Matrix
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title('Confusion Matrix - CNN with Dropout')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Modell speichern

Das beste Convolutional Neural Network Modell wird für den finalen Vergleich in Notebook 06 gespeichert.


In [ ]:
# Speichere das Dropout-CNN Modell (das beste)
model_path = models_dir / "cnn_best_model.h5"
modelCNN_dropout.save(model_path)

# Berechne Test Accuracy für Metadaten
test_loss_dropout, test_acc_dropout = modelCNN_dropout.evaluate(test_ds, verbose=0)

# Metadaten speichern
metadata = {
    "model_type": "Convolutional Neural Network (CNN with Dropout)",
    "architecture": "Conv2D(32) -> MaxPool -> Conv2D(64) -> MaxPool -> Dense(128) -> Output",
    "test_accuracy": float(test_acc_dropout),
    "test_loss": float(test_loss_dropout),
    "input_shape": (IMG_SIZE, IMG_SIZE, 3),
    "n_classes": num_classes,
    "regularization": "Dropout + Batch Normalization",
    "optimizer": "SGD with LR Schedule",
}

metadata_path = models_dir / "cnn_metadata.pkl"
with open(metadata_path, "wb") as f:
    pickle.dump(metadata, f)

print(f"CNN Modell und Metadaten gespeichert in {models_dir}")
print(f"  - Test Accuracy: {test_acc_dropout:.4f}")
print(f"  - Test Loss: {test_loss_dropout:.4f}")
print(f"  - Input Shape: {IMG_SIZE}x{IMG_SIZE} RGB")
print(f"  - Regularization: Dropout + Batch Normalization")
